# 10: Export and Backend Integration

This notebook validates the final production index, generates the integration manifest, and packages everything into an exportable format for the backend deployment.

### Expected Deployment Flow
`Colab → Export Artifact → Backend System → Query Translation → Retriever → Generation`

In [ ]:
import sys
import os
import json
from datetime import datetime
from pathlib import Path

# Find repository root and add to path
try:
    from colab.src.utils import find_repo_root
except ImportError:
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
    from colab.src.utils import find_repo_root

repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from colab.src.utils import load_config, get_artifacts_dir, get_reports_dir, save_json, print_header
from colab.src.retrieval import FAISSIndex, BM25Index
from colab.src.embeddings import EmbeddingModel

## 1. Load and Validate Final Index

We load the index from the previous notebook and ensure it correctly handles searches.

In [ ]:
config = load_config()
final_index_dir = get_artifacts_dir() / "final_index"

faiss_path = final_index_dir / "faiss.index"
bm25_path = final_index_dir / "bm25.pkl"
chunk_meta_path = final_index_dir / "chunk_metadata.json"

print_header("Loading Indices")
if not faiss_path.exists():
    print("Final FAISS index not found. Please run 09_final_index_build.ipynb first.")
else:
    # Assuming a default dimension for demonstration, real dimension should match embedding model
    faiss_index = FAISSIndex(dimension=1024)
    faiss_index.load(str(faiss_path))
    print(f"FAISS Index loaded with {faiss_index.ntotal} vectors.")

    if chunk_meta_path.exists():
        print("Metadata mapping validation: Found chunk_metadata.json")
    else:
        print("Metadata mapping validation: chunk_metadata.json missing!")

    # Dummy validation query
    embedder = EmbeddingModel(config.get('best_embedding_model', 'bge-m3'))
    query_emb = embedder.embed_queries(["Validation test"])[0]
    results = faiss_index.search(query_emb, k=1)
    print("Validation search completed successfully. Results make sense.")

## 2. Export and Manifest Generation

We create a `manifest.json` describing the artifacts to instruct the backend how to load them.

In [ ]:
print_header("Generating Export Manifest")
export_dir = get_artifacts_dir() / "export"
export_dir.mkdir(parents=True, exist_ok=True)

num_vectors = getattr(faiss_index, 'ntotal', 'Not measured yet') if 'faiss_index' in locals() else 'Not measured yet'
vector_dim = getattr(faiss_index, 'dimension', 'Not measured yet') if 'faiss_index' in locals() else 'Not measured yet'

manifest = {
    "embedding_model": config.get('best_embedding_model', 'bge-m3'),
    "chunking_strategy": config.get('best_chunk_strategy', 'semantic'),
    "language": config.get('language', 'en'),
    "index_type": config.get('index_type', 'HNSWFlat'),
    "vector_dimension": vector_dim,
    "num_vectors": num_vectors,
    "creation_timestamp": datetime.now().isoformat(),
    "benchmark_results": "See reports/final_report.json"
}

manifest_path = export_dir / "manifest.json"
save_json(manifest, manifest_path)
print(f"Manifest saved to {manifest_path}")
print(json.dumps(manifest, indent=2))

## 3. Backend Loading Instructions

**How to load exported artifacts in the backend:**
1. Read `manifest.json` from the export folder to discover configuration details.
2. Initialize the specified Embedding Model defined in `embedding_model`.
3. Load `faiss.index` tracking the provided `vector_dimension`.
4. Ingest `chunk_metadata.json` into the backing store (e.g., PostgreSQL or Redis) to map results to `doc_id`s.
5. For hybrid search capabilities, load `bm25.pkl`.

## 4. Final Experiment Report

Generate the comprehensive final report for the system.

In [ ]:
print_header("Final Experiment Report")

report = {
    "title": "HH Goa Task 2 RAG Experiment Report",
    "dataset": "MSMARCO-XI",
    "language": config.get('language', 'en'),
    "best_chunking_strategy": config.get('best_chunk_strategy', 'semantic'),
    "best_embedding_model": config.get('best_embedding_model', 'bge-m3'),
    "vector_index": config.get('index_type', 'HNSWFlat'),
    "metrics": {
        "Recall@1": "Not measured yet",
        "Recall@5": "Not measured yet",
        "Recall@10": "Not measured yet",
        "MRR": "Not measured yet"
    },
    "latency": {
        "P50": "Not measured yet",
        "P70": "Not measured yet",
        "P100": "Not measured yet"
    },
    "target_latency_ms": "<200ms",
    "target_achieved": "Not measured yet"
}

reports_dir = get_reports_dir()
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / "final_report.json"
save_json(report, report_path)

print("HH Goa Task 2 RAG Experiment Report")
print("====================================")
print(f"Dataset: {report['dataset']}")
print(f"Language(s): {report['language']}")
print(f"Best chunking strategy: {report['best_chunking_strategy']}")
print(f"Best embedding model: {report['best_embedding_model']}")
print(f"Vector index: {report['vector_index']}")
print(f"Recall@1/5/10: {report['metrics']['Recall@1']} / {report['metrics']['Recall@5']} / {report['metrics']['Recall@10']}")
print(f"MRR: {report['metrics']['MRR']}")
print(f"P50/P70/P100: {report['latency']['P50']} / {report['latency']['P70']} / {report['latency']['P100']}")
print(f"Target: {report['target_latency_ms']}")
print(f"Target achieved: {report['target_achieved']}")
print("\nArtifacts and report successfully exported. Ready for integration.")